# Train and inspect the pancreas models

Use this notebook to view a real training patch, read the **20-batch runtime check**, train either variant, and inspect validation predictions. The default cells load data and existing results; training and final testing have explicit switches.

The current audit retained **795 scans: 709 train, 36 validation, 50 test**. Both models use these same official partitions. Source metadata provides no patient mapping, so scan IDs remain grouping proxies.

There is **no augmentation** here. Randomly selecting a training patch is still part of sampling. CT is clipped to `[-200, 400]`, z-scored per volume, and kept on its downloaded approximately 1.5 mm grid. The network receives only CT; masks and derived boundaries are training targets.

Select the project `.venv/bin/python` as the notebook kernel. From the project directory, install notebook tools if needed with `python -m pip install -e ".[notebook]"`.

Before running, prepare the retained manifest and frozen splits with the commands in the project README. Generated data files and runtime reports remain local.


In [ ]:
from pathlib import Path
import copy
import csv
import json
import math
import sys

# Works when launched from the project directory or its notebooks/ directory.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Gated-Boundary-Encoding-CNN"]
ROOT = next((p for p in candidates if (p / "src/pancreas_seg").is_dir()
             and (p / "configs/boundary_attention.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from Gated-Boundary-Encoding-CNN or notebooks/.")
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import nibabel as nib
import numpy as np
import torch
from IPython.display import Markdown, display

from pancreas_seg.config import load_config
from pancreas_seg.data import PancreasDataset, load_pair, read_manifest, validate_splits
from pancreas_seg.engine import evaluate_checkpoint, train

print("Project:", ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU / CUDA unavailable")


**Choose the model and switches.** Start with `attention_unet`; later change `VARIANT` to `boundary_attention_unet` and rerun the cells below. The optional ungated control is `unet`.

Keep the same `RUN_NAME`, epoch count and base settings for the two planned variants. Choose a fresh run name for a new experiment. To continue an existing run, set `RESUME = True`. Leave `RUN_FINAL_TEST` false while selecting settings.


In [ ]:
VARIANT = "attention_unet"       # "boundary_attention_unet" or optional "unet"
DEVICE = "auto"                 # "auto", "cuda", or "cpu"
EPOCHS = 50
RUN_NAME = "no_augmentation_v1"

RUN_BENCHMARK = False            # optional new 20-batch run for BOTH planned variants
RUN_TRAINING = False             # explicitly enable when ready to train
RUN_VALIDATION = False           # evaluate the selected trained model on all 36 val scans
RUN_FINAL_TEST = False           # enable only after the recipe/checkpoint choices are frozen
RESUME = False
CHECKPOINT = None                # optional path to your own best.pt; None uses this run's best.pt

if VARIANT not in {"attention_unet", "boundary_attention_unet", "unet"}:
    raise ValueError("Choose one of the three model variants above.")
base_config = load_config(ROOT / "configs/boundary_attention.yaml")
base_config["augmentation"].update(affine_probability=0.0, elastic_probability=0.0,
                                    noise_probability=0.0, flip_axes=[])
base_config["data"]["boundary_halo"] = 1
base_config["training"]["epochs"] = EPOCHS
base_config["training"]["output_dir"] = str(ROOT / "runs" / RUN_NAME)
config = copy.deepcopy(base_config)
config["model"]["variant"] = VARIANT
run_dir = Path(config["training"]["output_dir"]) / VARIANT / f"seed_{config['seed']}"
checkpoint = Path(CHECKPOINT).expanduser().resolve() if CHECKPOINT else run_dir / "best.pt"
print("Selected variant:", VARIANT)
print("Run directory:", run_dir)
print("Checkpoint:", checkpoint)
print("Augmentation probabilities: affine=0, elastic=0, noise=0; flips=[]")


**Check the data going into the model.** A batch has shape `(N, C, D, H, W) = (1, 1, 96, 96, 96)`. Segmentation predicts two channels: background and pancreas. The full model also predicts one boundary channel.

One voxel of context on each side makes a temporary `98 × 98 × 98` patch. The boundary is derived before the final `96³` crop, so cutting through the organ does not create a false surface. The preview below deliberately centers on pancreas foreground to make it visible. Normal training keeps the configured foreground-sampling probability.


In [ ]:
records = read_manifest(config["data"]["manifest"])
splits = json.loads(Path(config["data"]["splits"]).read_text())
validate_splits(records, splits)
print("Cases:", len(records), {name: len(splits[name]) for name in ("train", "val", "test")})
patches_per_epoch = len(splits["train"]) * config["data"]["patches_per_case"]
batches_per_epoch = math.ceil(patches_per_epoch / config["training"]["batch_size"])
updates_per_epoch = math.ceil(batches_per_epoch / config["training"]["accumulation_steps"])
print(f"Per epoch: {patches_per_epoch} patches, {batches_per_epoch} batches, {updates_per_epoch} optimizer updates.")
print("A 20-batch timing check covers batches, not 20 epochs or 20 full-volume predictions.")


def show_slices(image, target, *, boundary=None, prediction=None, title=""):
    coordinates = np.argwhere(target > 0)
    center = np.median(coordinates, axis=0).astype(int) if len(coordinates) else np.array(target.shape) // 2
    limits = np.percentile(image, [1, 99])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, ax in enumerate(axes):
        section = [slice(None)] * 3
        section[axis] = int(center[axis])
        section = tuple(section)
        ax.imshow(image[section].T, origin="lower", cmap="gray", vmin=limits[0], vmax=limits[1])
        colors = np.zeros((*target[section].T.shape, 4))
        colors[target[section].T > 0] = [1.0, 0.8, 0.0, 0.35]
        if boundary is not None:
            colors[boundary[section].T > 0] = [1.0, 0.1, 0.1, 0.85]
        ax.imshow(colors, origin="lower")
        if prediction is not None:
            pred_colors = np.zeros_like(colors)
            pred_colors[prediction[section].T > 0] = [0.0, 0.9, 1.0, 0.45]
            ax.imshow(pred_colors, origin="lower")
        ax.set_title(("Sagittal", "Coronal", "Axial")[axis])
        ax.set_axis_off()
    legend = [Patch(color="gold", label="Ground-truth pancreas")]
    if boundary is not None:
        legend.append(Patch(color="red", label="Derived inner boundary"))
    if prediction is not None:
        legend.append(Patch(color="cyan", label="Model prediction"))
    fig.legend(handles=legend, loc="lower center", ncol=len(legend))
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    plt.show()

preview_record = next(r for r in records if r["case_id"] == splits["train"][0])
preview_config = copy.deepcopy(config)
preview_config["data"]["foreground_probability"] = 1.0
sample = PancreasDataset([preview_record], preview_config, training=True)[0]
print("Training case:", sample["case_id"], "| CT shape:", tuple(sample["image"].shape))
show_slices(sample["image"][0].numpy(), sample["label"][0].numpy(),
            boundary=sample["boundary"][0].numpy(), title="Training patch — no augmentation")


**Read the measured 20-batch check.** This compares elapsed time and GPU memory for the same training patches, including data loading. Twenty batches provide a runtime estimate; they do not measure segmentation quality.

The training-epoch projection excludes full-volume validation and checkpoint saving. The boundary variant has an extra loss term, so its total loss is not directly comparable to the baseline's total loss. Compare the segmentation-loss curves as a plumbing check.

A fresh clone has no saved runtime report. Set `RUN_BENCHMARK=True` to create a local 20-batch measurement; otherwise the notebook skips the runtime plots when no report exists.


In [ ]:
benchmark_report = None
if RUN_BENCHMARK:
    from pancreas_seg.benchmark import benchmark_training
    benchmark_report = benchmark_training(
        base_config, variants=("attention_unet", "boundary_attention_unet"), batches=20,
        device_name=DEVICE, output_dir=ROOT / "runs" / f"{RUN_NAME}_benchmark")
else:
    report_paths = [ROOT / "runs" / f"{RUN_NAME}_benchmark" / "report.json",
                    ROOT / "runs/runtime_smoke/report.json",
                    ROOT / "verification/runtime_smoke20.json"]
    report_path = next((p for p in report_paths if p.is_file()), None)
    if report_path:
        benchmark_report = json.loads(report_path.read_text())
        print("Loaded measured report:", report_path)
    else:
        print("No saved 20-batch report found. Set RUN_BENCHMARK=True for a new measurement.")

if benchmark_report:
    print("Device:", benchmark_report["device"], "| AMP:", benchmark_report["amp"],
          "| batches requested:", benchmark_report["requested_batches"])
    rows = ["| Variant | Batches | Mean seconds/batch | Projected train epoch (min) | Peak GPU allocated (GiB) |",
            "|---|---:|---:|---:|---:|"]
    for variant, result in benchmark_report["variants"].items():
        mean = result.get("mean_batch_seconds")
        epoch = result.get("projected_train_epoch_seconds")
        memory = result.get("peak_cuda_allocated_gib")
        rows.append(f"| {variant} | {result['batches_completed']} | "
                    f"{f'{mean:.3f}' if mean is not None else 'n/a'} | "
                    f"{f'{epoch / 60:.1f}' if epoch is not None else 'n/a'} | "
                    f"{f'{memory:.2f}' if memory is not None else 'n/a'} |")
    display(Markdown("\n".join(rows)))


In [ ]:
if benchmark_report:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for variant, result in benchmark_report["variants"].items():
        rows = result.get("batch_rows", [])
        if not rows:
            continue
        batch_numbers = [r["batch"] for r in rows]
        axes[0].plot(batch_numbers, [r["seg_loss"] for r in rows], marker=".", label=variant)
        axes[1].plot(batch_numbers, [r["batch_seconds"] for r in rows], marker=".", label=variant)
    axes[0].set(title="20-batch segmentation loss", xlabel="Training batch", ylabel="Soft Dice loss")
    axes[1].set(title="Measured time including loading", xlabel="Training batch", ylabel="Seconds")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    plt.show()


**Train the selected variant.** Set `RUN_TRAINING=True` above and execute this cell when ready. `EPOCHS` means complete passes over the training cases. At the current defaults, one epoch has 709 batches and 355 optimizer updates because gradients accumulate over two batches.

The trainer selects `best.pt` using validation Dice; it also saves `last.pt`, configuration, history and frozen splits. It refuses to overwrite an existing run unless you explicitly resume. To train the other planned variant, change `VARIANT` and rerun from the control cell with the same other settings.


In [ ]:
if RUN_TRAINING:
    resume_path = str(run_dir / "last.pt") if RESUME else None
    run_dir = train(config, device_name=DEVICE, resume=resume_path)
    checkpoint = run_dir / "best.pt"
    print("Training complete. Validation-selected checkpoint:", checkpoint)
else:
    print("Training is disabled. Set RUN_TRAINING=True in the control cell to start.")


**Inspect training history.** These plots load any completed baseline/full-model runs under the chosen `RUN_NAME`. The boundary curve appears only for the full model. Validation occurs at the configured interval, so its curve has fewer points.


In [ ]:
histories = {}
for variant in ("attention_unet", "boundary_attention_unet", "unet"):
    history_path = Path(base_config["training"]["output_dir"]) / variant / f"seed_{config['seed']}" / "history.csv"
    if history_path.is_file():
        with history_path.open(newline="") as stream:
            histories[variant] = list(csv.DictReader(stream))

if histories:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    for variant, rows in histories.items():
        epochs = [int(r["epoch"]) for r in rows]
        axes[0].plot(epochs, [float(r["seg_loss"]) for r in rows], label=variant)
        if variant == "boundary_attention_unet":
            axes[1].plot(epochs, [float(r["boundary_loss"]) for r in rows], label=variant)
        val_rows = [r for r in rows if r["val_dice"] != ""]
        axes[2].plot([int(r["epoch"]) for r in val_rows], [float(r["val_dice"]) for r in val_rows],
                     marker="o", label=variant)
    for ax, title, ylabel in zip(axes, ["Training segmentation loss", "Training boundary loss", "Validation Dice"],
                                ["Soft Dice loss", "Weighted BCE loss", "Dice"]):
        ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
        ax.grid(alpha=0.25)
        if ax.lines:
            ax.legend()
    axes[2].set_ylim(0, 1)
    fig.tight_layout()
    plt.show()
else:
    print("No training history yet for", RUN_NAME)


**Evaluate validation scans and view a prediction.** Set `RUN_VALIDATION=True` after training. This reconstructs full 3D volumes with sliding windows, saves per-case metrics and exports predictions. It can take substantially longer than processing 20 training patches.

The overlay uses the checkpoint's own preprocessing settings and saved validation prediction grid. Predictions are not automatically restored to the original acquisition grid. Only validation examples appear here.


In [ ]:
validation_dir = checkpoint.parent / "evaluation_val"
if RUN_VALIDATION:
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Train a model or set CHECKPOINT first: {checkpoint}")
    validation_summary = evaluate_checkpoint(checkpoint, split="val", device_name=DEVICE, save_predictions=True)
    print(json.dumps(validation_summary, indent=2))
elif (validation_dir / "summary.json").is_file():
    print("Saved validation summary:")
    print((validation_dir / "summary.json").read_text())
else:
    print("No validation evaluation yet. Set RUN_VALIDATION=True after training.")


In [ ]:
prediction_files = sorted((validation_dir / "predictions").glob("*.nii.gz"))
if checkpoint.is_file() and prediction_files:
    # Load only checkpoints produced by your own runs.
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    prediction_path = prediction_files[0]
    case_id = prediction_path.name.removesuffix(".nii.gz")
    if case_id not in state["splits"]["val"]:
        raise ValueError("The displayed prediction must belong to validation.")
    frozen_records = read_manifest(state["config"]["data"]["manifest"])
    record = next(r for r in frozen_records if r["case_id"] == case_id)
    pair = load_pair(record, state["config"]["data"])
    prediction_image = nib.load(prediction_path)
    if prediction_image.shape != pair["label"].shape[1:] or not np.allclose(prediction_image.affine, pair["affine"]):
        raise ValueError("Prediction and preprocessed CT/mask grids differ.")
    prediction = np.asarray(prediction_image.dataobj) > 0
    show_slices(pair["image"][0], pair["label"][0], prediction=prediction,
                title=f"{VARIANT} — validation case {case_id}")
    del state, pair, prediction
else:
    print("A saved full-volume validation prediction is needed for this overlay.")


**Final held-out test.** Keep this off until model settings and checkpoint-selection rules are fixed. Then evaluate each variant once using its validation-selected checkpoint. The 50 retained test cases are reserved for this final comparison.

Record Dice, precision/recall, boundary F1 and HD95 together with parameter count, runtime and memory. A short runtime check or falling training loss does not establish improved segmentation.


In [ ]:
if RUN_FINAL_TEST:
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Missing validation-selected checkpoint: {checkpoint}")
    test_summary = evaluate_checkpoint(checkpoint, split="test", device_name=DEVICE, save_predictions=False)
    print(json.dumps(test_summary, indent=2))
else:
    print("Final test remains disabled while the training recipe is being chosen.")
